# Mr. Beast Continues... to Systems of Equations!

In [ ]:
suppressPackageStartupMessages(library(coursekata))

# format mrbeast15
mrbeast15 <- read.csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vTkAp9lm97EsZsmwLG_3OeowyJ5W70GOjlBQE3GljlkpHzZhSNQrMSXEy2SYpmDZjMWSMXmNoDRG6em/pub?gid=608407973&single=true&output=csv") %>%
  mutate(pull_date = as.Date(pull_date),
         pub_date = as.Date(pub_date))

# the date Mr. Beast posted first video
date_start <- as.Date("2012-02-20")

<div class="alert alert-info">
    
<center>Paper Notes</center>

https://docs.google.com/document/d/1apUdGgN8cUsI0Hw7Zxri4cXJ41A_QYm8R8jCqb40dNc/edit?usp=sharing
    
</div>

## `mrbeast15` Data Frame (Recap)

The data frame `mrbeast15` has the 15 highest viewed Mr. Beast videos (as of September 2023). The data are a subset of a giant data set scraped daily from youtube by a data scientist and made available on [Kaggle](https://www.kaggle.com/datasets/robikscube/mrbeast-youtube-stats-daily), an AI and machine learning community. There are 197 rows of data. Each row represents a video's data on a particular day because views, likes, and comment counts change every day.

There are 12 variables in this data frame:

- `id` a random string of characters, numbers, and symbols representing a particular video
- `title` the title of the video
- `duration_sec` length of video in seconds
- `pull_date` the date when the data (e.g., like count) were scraped from youtube 
- `pub_date` the date the video was published to youtube
- `days_since_pub` how many days since the video was published 
- `days_since_start` how many days since Mr. Beast started this channel (in 2012); note that the data started being scraped 3607 after Mr. Beast started the channel so that is the lowest number
- `view_m` number of views in millions (i.e., 300 means 300 million). Note that youtube defines a "view" as each time a viewer intentionally initiates the playing of a video on their device and watches for at least 30 seconds. 
- `like_m` number of likes in millions (i.e., 15 means 15 million)  
- `comment_k` number of comments in thousands (i.e., 30 means 30,000) 
- `big_num_title` 1 if there is a big number (greater than 1,000) in the title of the video and 0 otherwise
- `short` 1 if this video is a youtube short and 0 if not

## 1.0: Where we ended

When we last ended our exploration of Mr. Beast data, we saw that YouTube shorts might have a different pattern of views compared to non-shorts. 


In [ ]:
gf_point(view_m ~ days_since_start, data = mrbeast15, shape = ~short, color = ~short) 

**1.1:** Should we use a piecewise function to model this word equation: **view_m = short + days_since_start + other stuff**? Why or why not?

## 2.0: Multiple Linear Functions

<div class="alert alert-info">
    
<center>Paper Notes</center>

Difference between piecewise functions and systems of equations.
    
</div>

**2.1:** The strategy is just to make a best-fitting model for nonshorts and also to do the same for shorts. 

Here's some code that finds the best-fitting parameter estimates for nonshorts. Create a custom function for nonshorts. 

Duplicate the same approach for shorts.

In [ ]:
# Y ~ X
lm(view_m ~ days_since_start, data = filter(mrbeast15, short == 0))
nonshort_function <- function(X){ -743.072 + 0.244*X }


lm(view_m ~ days_since_start, data = filter(mrbeast15, short == 1))
short_function <- function(X){ -4964.330 + 1.302*X }

**2.2:** Try putting both functions on our scatter plot below. (Note if you want to match colors, you might try `"lightseagreen"` and `"purple"`.)

In [ ]:

gf_point(view_m ~ days_since_start, data = mrbeast15, color = ~short, shape = ~short) %>%
  gf_function(nonshort_function, color = "lightseagreen") %>%
  gf_function(short_function, color = "purple")

## 3.0: Evaluate Models: How much error has been reduced?

In statistics, we actually write these two separate but related functions as a single multivariate function, now with more than one X (we call them X1 and X2). 

You can think of the word equation as: **Y = X1 + X2 + other stuff**

In `mrbeast15` data, it is: **view_m = short + days_since_start + other stuff**

Here we've written some code to save each variable as it's own vector.

In [ ]:
Y <- mrbeast15$view_m
X1 <- mrbeast15$short
X2 <- mrbeast15$days_since_start

**3.1:** We can make a mega-multivariate-function out of our two simpler functions. (Note: Unfortunately, you cannot graph this mega-function! That's why it's helpful to make the two smaller ones.)

In [ ]:
mega_function <- function(X1, X2){ 
  ifelse(X1 == 0,
        nonshort_function(X2),
        short_function(X2))
}

mega_function(0, 4200)
mega_function(1, 4200)

<div class="alert alert-info">
    
<center>Paper Notes</center>

**3.2:** Draw a few residuals from the mega-multivariate function.
    
</div>

**3.3:** The beauty of the mega-function is that you can use it to calculate PRE (and everything else) just like we would for a simpler function.

In [ ]:
# edit this code to calculate PRE with the mega_function
sse <- sse(Y, mega_function(X1, X2))
sst <- sse(Y, mean(Y))

(sst - sse) / sst

## Appendix

How I made the figures and print outs.

In [ ]:
lm(view_m ~ days_since_start, data = filter(mrbeast15, short == 0))
lm(view_m ~ days_since_start, data = filter(mrbeast15, short == 1))

nonshort_function <- function(X){-743.072 + 0.244*X}
short_function <- function(X){-4964.330 + 1.302*X}
null_function <- function(X){mean(Y)}

In [ ]:
# data visualizations
gf_point(view_m ~ days_since_start, data = mrbeast15, shape = ~(days_since_start < 3968), show.legend = FALSE) %>%
  gf_vline(xintercept = 3968, linetype = "dashed", linewidth = .5)

gf_point(view_m ~ days_since_start, data = mrbeast15, shape = ~short, show.legend = FALSE) %>%
  gf_model(lm(view_m ~ days_since_start * short, data = mrbeast15)) +
  theme_bw()

gf_point(view_m ~ days_since_start, data = mrbeast15, shape = ~short, show.legend = FALSE) %>%
  gf_facet_wrap(~short) %>%
  gf_model(lm(view_m ~ days_since_start * short, data = mrbeast15))+
  theme_bw()

gf_point(view_m ~ days_since_start, data = mrbeast15, shape = ~short, show.legend = FALSE) %>%
  gf_facet_wrap(~short) %>%
  gf_model(lm(view_m ~ days_since_start * short, data = mrbeast15))+
  theme_bw()


In [ ]:

gf_point(view_m ~ days_since_start, data = mrbeast15, shape = ~short, show.legend = FALSE) %>%
  gf_model(lm(view_m ~ NULL, data = mrbeast15))%>%
  gf_model(lm(view_m ~ days_since_start * short, data = mrbeast15), alpha = .1)+
  theme_bw()

gf_point(view_m ~ days_since_start, data = mrbeast15, shape = ~short, show.legend = FALSE) %>%
  gf_facet_wrap(~short) %>%
  gf_model(lm(view_m ~ NULL, data = mrbeast15))%>%
  gf_model(lm(view_m ~ days_since_start * short, data = mrbeast15), alpha = .1)+
  theme_bw()